# Exercises on NumPy basics - solutions

In [2]:
import numpy as np

## Creating arrays and data types

### Exercise 1

In [ ]:
L = 2.        # m
n_int = 10    # number of intervals

# 1. n_int intervals means n_int + 1 nodes
x = np.linspace(0., L, n_int + 1)
print(x)
print(x.shape, x.size, x.dtype)

# 2. An array of a constant value, in two equivalent ways
area = np.full(n_int + 1, 2.5e-3)
print(area)
print(np.allclose(area, 2.5e-3*np.ones(n_int + 1)))

# An array of zeros, to be filled later with the results
results = np.zeros(n_int + 1)
print(results.ndim, area.ndim)

# 3. Vectorized: the expression is applied to all the nodes at once
alpha = 1.2e-5
delta_T = 60.
u = alpha*delta_T*x
print(f"displacement of the free end: {1000*u[-1]:.3f} mm")

# 4. arange excludes the stop value, so the last node (x = L) is missing
x_arange = np.arange(0., L, L/n_int)
print(x_arange, x_arange.shape)

### Exercise 2

In [ ]:
readings = np.array([1.2345e-3, 2.1017e-3, 0.9876e-3, 3.4562e-3])

# 1. A scalar multiplies every element (broadcasting)
micro = 1e6*readings
print(micro, micro.dtype)

# 2. astype TRUNCATES, so the correct conversion needs np.round first
print(micro.astype(np.int64))            # WRONG: 2101.7 becomes 2101
micro_int = np.round(micro).astype(np.int64)
print(micro_int)                         # correct: 2101.7 becomes 2102
# Note that 1234.5 is rounded to 1234: np.round, like round(), sends the halfway
# cases to the nearest EVEN number (see Part 2)

# 3. Data types
print(readings.dtype, micro_int.dtype)

# 4. Back to strain units
print(np.max(np.abs(readings - micro_int/1e6)))

## Indexing, slicing, views and copies

### Exercise 3

In [ ]:
data = np.array([[0.000,   0., 20.0],
                 [0.001, 210., 21.2],
                 [0.002, 415., 23.4],
                 [0.003, 500., 26.1],
                 [0.004, 515., 30.5]])

# 1. All the rows, one column
strain = data[:,0]
stress = data[:,1]
print(strain)
print(stress)

# 2. Indexing a dimension makes it disappear, slicing it keeps it
print(data[0,:], data[0,:].shape, data[0,:].ndim)
print(data[0:1,:].shape, data[0:1,:].ndim)    # still two-dimensional: a 1x3 matrix

# 3. All the rows, the first two columns
print(data[:,:2])

# 4. Last row, second column
print(data[-1,1])

# 5. Writing through an index modifies the array in place
data[:,2] += 0.5
print(data)

### Exercise 4

In [ ]:
data = np.array([[0.000,   0., 20.0],
                 [0.001, 210., 21.2],
                 [0.002, 415., 23.4],
                 [0.003, 500., 26.1],
                 [0.004, 515., 30.5]])

# 1. Slicing an array returns a VIEW: stress and data share the same memory
stress = data[:,1]
stress[0] = 1000.
print(data[:,1])    # the table has changed too!

# 2. copy() returns an independent array
data[0,1] = 0.
stress_copy = data[:,1].copy()
stress_copy[0] = 1000.
print(data[:,1])    # this time the table is unchanged

# 3. The multiplication builds a new array, so the table is safe
def scaled_stress(table, factor):
    return factor*table[:,1]

print(scaled_stress(data, 2.))
print(data[:,1])

## Element-wise operations

### Exercise 5

In [ ]:
s1 = np.array([120., -80., 200.,  50.])   # MPa
s2 = np.array([ 40., -150., -60.,  50.])  # MPa

# 1. np.sqrt is applied element by element
s_eq = np.sqrt(s1**2 - s1*s2 + s2**2)
print(s_eq)

# 2. and 3. A scalar divided by an array gives an array
sigma_y = 250.
safety = sigma_y/s_eq
print(np.round(safety, 2))

# 4. For an equibiaxial state the equivalent stress equals the principal one
print(np.isclose(s_eq[3], s1[3]))

### Exercise 6

In [ ]:
A0 = 5.               # mm
zeta = 0.05
omega_n = 2*np.pi*20  # rad/s

# 1. np.exp is applied to the whole array
t = np.linspace(0., 0.5, 6)
A = A0*np.exp(-zeta*omega_n*t)
print(A)

# 2. log(A) = log(A0) - zeta*omega_n*t is linear in t, so its successive
# differences (computed by slicing) are all equal
log_A = np.log(A)
differences = log_A[1:] - log_A[:-1]
print(differences)
print(np.allclose(differences, differences[0]))

# 3. The slope is -zeta*omega_n, and the time step is constant
slope = differences[0]/(t[1] - t[0])
print(np.isclose(-slope/omega_n, zeta))

### Exercise 7

In [ ]:
theta = np.linspace(0., 2*np.pi, 8, endpoint=False)
holes = np.hstack((60*np.cos(theta).reshape(-1,1), 60*np.sin(theta).reshape(-1,1)))

# 1. One column each, then the polar coordinates
x, y = holes[:,0], holes[:,1]
radii = np.sqrt(x**2 + y**2)
angles = np.degrees(np.arctan2(y, x))
print(radii)
print(angles)

# 2. All the radii are equal: never compare floats with ==
print(np.allclose(radii, radii[0]))
print(f"bolt circle radius: {radii[0]:.1f} mm")

# 3. The pitch is the difference between two consecutive angles
print(f"angular pitch: {angles[1] - angles[0]:.1f} degrees")

## Comparisons, masks and filtering

### Exercise 8

In [3]:
rng = np.random.default_rng(seed=3)
diameters = rng.normal(5.0, 0.02, 200)   # mm

nominal, tol = 5.0, 0.03

# 1. Two conditions combined with &, each one inside parentheses
accepted = (diameters >= nominal - tol) & (diameters <= nominal + tol)
rejected = ~accepted      # ~ is the element-wise "not"

# 2. Summing a mask counts its True values
n_rejected = np.sum(rejected)
print(n_rejected)
print(f"rejected: {100*n_rejected/diameters.size:.1f}%")

# 3. Global checks
print(np.any(diameters > 5.05), np.all(diameters > 4.90))

# 4. The mask selects the elements of interest
print(diameters[rejected])
print(f"largest deviation: {1000*np.max(np.abs(diameters - nominal)):.1f} micrometres")

30
rejected: 15.0%
True True
[5.04081838 4.9488867  4.95960028 5.06645999 5.03091642 5.03870176
 4.94343675 4.9666276  4.95012211 4.96722286 4.95931665 4.95683989
 5.0318128  4.9656363  5.03363651 4.96539973 5.05100069 4.95920992
 4.95441947 5.04095772 5.03519853 5.03009895 5.04926264 4.96947682
 5.03390415 4.9581903  5.03219325 5.04321813 5.04351151 5.03231364]
largest deviation: 66.5 micrometres


### Exercise 9

In [ ]:
stress = np.array([120., 135., -999., 150., 168., -999., 210., 260., 245.])   # MPa

# 1. The mask of the valid samples
valid = stress != -999.
mean_valid = stress[valid].mean()
print(f"{np.sum(valid)} valid samples, mean {mean_valid:.1f} MPa")

# 2. where(condition, if_true, if_false), element by element
cleaned = np.where(valid, stress, mean_valid)
print(cleaned)

# 3. where() with the condition only returns the indices
yielded = np.where(valid & (stress > 250.))[0]
print(yielded)
print(yielded[0] if yielded.size > 0 else -1)

## Reductions and the axis argument

### Exercise 10

In [ ]:
acc = np.array([[0.12, 0.31, 0.08],
                [0.15, 0.29, 0.11],
                [0.22, 0.35, 0.09],
                [0.19, 0.41, 0.12],
                [0.31, 0.38, 0.15],
                [0.28, 0.44, 0.13]])

# 1. One value per sensor (per column): the rows disappear, hence axis=0
print(acc.mean(axis=0))
print(acc.std(axis=0))
print(acc.min(axis=0), acc.max(axis=0))

# 2. One value per time step (per row): the columns disappear, hence axis=1
print(acc.max(axis=1))

# 3. Maximum of each sensor, then the position of the largest
peaks = acc.max(axis=0)
print(peaks, peaks.argmax())

# 4. Maximum of each time step, then the position of the largest
print(acc.max(axis=1).argmax())

# 5. The mean of the squares, column by column, then the square root
print(np.sqrt((acc**2).mean(axis=0)))

## Broadcasting

### Exercise 11

In [4]:
nodes = np.array([[0., 0.],
                  [1., 0.],
                  [2., 0.],
                  [1., 1.]])

# 1. (4,1,2) minus (1,4,2) gives, by broadcasting, the (4,4,2) array of all the
# coordinate differences. Summing the squares along the last axis gives the (4,4) matrix
diff = nodes[:,np.newaxis,:] - nodes[np.newaxis,:,:]
D = np.sqrt((diff**2).sum(axis=2))
print(np.round(D, 3))

# 2. Symmetry and zero diagonal
print(np.allclose(D, D.T))
print(np.allclose(np.diag(D), 0.))

# 3. Largest distance between two nodes, and mean distance from node 0
print(f"largest distance: {D.max():.3f} m")
print(f"mean distance from node 0: {D[0,1:].mean():.3f} m")

# BONUS: x becomes a column of shape (n,1), y stays a row of shape (m,)
def cauchy_matrix(x, y):
    return 1/(x.reshape((-1,1)) - y)

x = np.array([1., 2., 3.])
y = np.array([4., 5., 6., 7.])
print(cauchy_matrix(x, y).shape)
print(cauchy_matrix(x, y))

[[0.    1.    2.    1.414]
 [1.    0.    1.    1.   ]
 [2.    1.    0.    1.414]
 [1.414 1.    1.414 0.   ]]
True
True
largest distance: 2.000 m
mean distance from node 0: 1.471 m
(3, 4)
[[-0.33333333 -0.25       -0.2        -0.16666667]
 [-0.5        -0.33333333 -0.25       -0.2       ]
 [-1.         -0.5        -0.33333333 -0.25      ]]


## Shape manipulation and concatenation

### Exercise 12

In [7]:
# 1. reshape(-1,1) turns a vector into a column, hstack puts the columns side by side
t = np.linspace(0., 0.2, 5)          # one period of a 5 Hz signal
u = 0.05*np.sin(2*np.pi*5*t)
table = np.hstack((t.reshape(-1,1), u.reshape(-1,1)))
print(table)
print(table.shape)

# 2. vstack stacks along the rows
both = np.vstack((table, table))
print(both.shape)

# 3. ravel flattens, one row after the other
print(table.ravel(), table.ravel().shape)

# 4. The transpose exchanges rows and columns
print(table.T)
print(table.T.shape)
print(np.allclose(table.T, np.transpose(table)))

# 5. hstack is a short-hand for concatenate along the columns (axis=1)
same = np.concatenate((t.reshape(-1,1), u.reshape(-1,1)), axis=1)
print(np.allclose(table, same))

# BONUS
rng = np.random.default_rng(seed=0)
M1 = rng.integers(1, 5, (2,2))    # 5 is excluded, so the values are between 1 and 4
M2 = rng.integers(1, 5, (2,2))
M3 = rng.integers(1, 5, (4,2))
concatenated = np.hstack((M1, M2))     # shape (2,4)
print(concatenated)
print(concatenated @ M3)               # (2,4) @ (4,2) -> (2,2)

[[ 0.0000000e+00  0.0000000e+00]
 [ 5.0000000e-02  5.0000000e-02]
 [ 1.0000000e-01  6.1232340e-18]
 [ 1.5000000e-01 -5.0000000e-02]
 [ 2.0000000e-01 -1.2246468e-17]]
(5, 2)
(10, 2)
[ 0.0000000e+00  0.0000000e+00  5.0000000e-02  5.0000000e-02
  1.0000000e-01  6.1232340e-18  1.5000000e-01 -5.0000000e-02
  2.0000000e-01 -1.2246468e-17] (10,)
[[ 0.0000000e+00  5.0000000e-02  1.0000000e-01  1.5000000e-01
   2.0000000e-01]
 [ 0.0000000e+00  5.0000000e-02  6.1232340e-18 -5.0000000e-02
  -1.2246468e-17]]
(2, 5)
True
True
[[4 3 2 1]
 [3 2 1 1]]
[[23 37]
 [16 26]]


## Sorting, searching and unique values

### Exercise 13

In [8]:
def get_largest(a, n):
    # n must be strictly positive
    assert n > 0
    # np.sort gives the ascending order: reverse it, then take the first n elements
    return np.sort(a)[::-1][:n]

loads = np.array([1., 4., 2., 4.3, 5., 12., 7., 8., 2.])
print(get_largest(loads, 5))

[12.   8.   7.   5.   4.3]


### Exercise 14

In [9]:
names = np.array(["steel", "aluminium", "titanium", "copper", "magnesium", "CFRP"])
density = np.array([7850., 2700., 4500., 8960., 1740., 1600.])    # kg/m^3
strength = np.array([355., 270., 880., 210., 200., 600.])         # MPa

# 1. Element-wise division
specific = strength/density
print(np.round(specific, 4))

# 2. argsort gives the ascending order: reverse it to get the descending one,
# then use the resulting indices on both arrays (fancy indexing)
order = np.argsort(specific)[::-1]
print(names[order][:3])
print(np.round(specific[order][:3], 4))

# 3. argmin returns the position of the minimum
print(names[specific.argmin()])

[0.0452 0.1    0.1956 0.0234 0.1149 0.375 ]
['CFRP' 'titanium' 'magnesium']
[0.375  0.1956 0.1149]
copper


### Exercise 15

In [10]:
def unique_rows(A):
    # axis=0 makes unique work on entire rows instead of single elements
    return np.unique(A, axis=0)

tests = np.array([[100., 10.],
                  [150., 10.],
                  [100., 10.],
                  [150., 20.],
                  [100., 10.]])

conditions = unique_rows(tests)
print(conditions)
print(f"{conditions.shape[0]} distinct test conditions out of {tests.shape[0]} tests")

[[100.  10.]
 [150.  10.]
 [150.  20.]]
3 distinct test conditions out of 5 tests


### Exercise 16

In [ ]:
def first_occurrence(x, y):
    # where() returns a tuple with one array of indices per dimension
    occurrences = np.where(x == y)[0]
    return occurrences[0] if occurrences.size > 0 else -1

x = np.array([1, 2, 4, 21, 5, 2, 3])
print(first_occurrence(x, 2))
print(first_occurrence(x, 7))

def closest_element(array, number):
    # the element with the smallest distance from "number"
    index_closest = np.abs(array - number).argmin()
    return float(array[index_closest])

catalogue = np.array([6., 8., 10., 12., 16., 20., 25., 32.])
print(closest_element(catalogue, 13.7))

## Linear algebra

### Exercise 17

In [11]:
K = 1e3*np.array([[200., -100.,    0.],
                  [-100.,  200., -100.],
                  [   0., -100.,  100.]])   # N/m
f = np.array([0., 0., 500.])                # N

# 1. Never invert the matrix to solve a system: use solve()
u = np.linalg.solve(K, f)
print(np.round(1000*u, 4), "mm")

# 2. The check must be done with allclose, not with ==
print(np.allclose(K @ u, f))
print(np.allclose(np.dot(K, u), K @ u))   # np.dot and @ agree for matrices and vectors
print(np.linalg.norm(K @ u - f))

# 3. Symmetry of the stiffness matrix
print(np.allclose(K, K.T))

# 4. The inverse, only as a check
K_inv = np.linalg.inv(K)
print(np.allclose(K @ K_inv, np.eye(3)))

# BONUS: eigh is for symmetric matrices and returns ascending eigenvalues
eigenvalues, eigenvectors = np.linalg.eigh(K)
frequencies = np.sqrt(eigenvalues)/(2*np.pi)
print(np.round(frequencies, 2), "Hz")

[ 5. 10. 15.] mm
True
True
7.866737095637891e-14
True
True
[22.4  62.76 90.69] Hz


## Reading and writing arrays

### Exercise 18

In [ ]:
# The files are written in the folder that contains this notebook
np.save('displacements.npy', u)
np.save('stiffness.npy', K)

u_loaded = np.load('displacements.npy')
K_loaded = np.load('stiffness.npy')

print(u_loaded)
print(np.allclose(u, u_loaded), np.allclose(K, K_loaded))